In [ ]:
import pandas as pd

In [ ]:
results = pd.read_csv('usgsid_comid_results_2025.csv', dtype={'USGSID': str, 'COMID': str})

In [ ]:
results2 = pd.read_csv('site_info_active.csv', dtype={'site_no': str})

In [ ]:
print(results.head())

In [ ]:
print(results2.head())

In [ ]:
import geopandas as gpd

sites_gdf = gpd.GeoDataFrame(
    results2,
    geometry=gpd.points_from_xy(results2['dec_long_va'],results2['dec_lat_va']),
    crs="EPSG:4326"
)

In [ ]:
print(sites_gdf.head())

In [ ]:
print(results['USGSID'].str.len().value_counts())
print(sites_gdf['site_no'].str.len().value_counts())

In [ ]:
sites_gdf.rename(columns={'site_no': "USGSID"}, inplace=True)

In [ ]:
print(sites_gdf.head())

In [ ]:
df_merged = pd.merge(sites_gdf, results, on='USGSID', how='left')

In [ ]:
print(df_merged.tail())

In [ ]:
df_merged.shape

In [ ]:
df_merged['COMID'].isna().sum()

In [ ]:
df_merged = df_merged.drop(columns=['dec_lat_va', 'dec_long_va', 'alt_va', 'parm_cd'])

In [ ]:
print(df_merged.columns)

In [ ]:
# Keep only CONUS gages — drop Alaska, Hawaii, and Pacific/Caribbean territories.
# .cx is GeoPandas' bounding-box slice: df.cx[xmin:xmax, ymin:ymax] = [West:East, South:North].
# This box matches the app's map extent, so data scope and view scope stay consistent.
before = len(df_merged)
df_merged = df_merged.cx[-125:-66, 24:50]
print(f"CONUS filter: {before} -> {len(df_merged)} ({before - len(df_merged)} dropped)")


In [ ]:
# Store the prefixed USGS id form per Sudip's schema decision: "USGS-01646500".
# The guard prevents double-prefixing if the cell is re-run without reloading data.
mask = ~df_merged['USGSID'].astype(str).str.startswith('USGS-')
df_merged.loc[mask, 'USGSID'] = 'USGS-' + df_merged.loc[mask, 'USGSID'].astype(str)


In [ ]:
df_merged.to_file('../tethysapp/hydro_correlation_tool/public/data/merged_gages.geojson', driver='GeoJSON')